# 第3课：Layer 类与 Network 类

**学习目标：**
- 用面向对象的方式封装神经网络层
- 实现 `Layer` 类（权重初始化 + 前向传播）
- 实现 `Network` 类（自动堆叠多层）
- 理解 Python 类的 `__init__` 方法

---

上一课我们手动写了每层的矩阵运算。当网络变深时，逐层手写会非常冗长。本课将计算逻辑封装成可复用的类，让网络构建变得简洁。

## 3.1 函数式多层网络

在封装类之前，先看看不用类时写多层网络有多麻烦。以 `[2, 3, 4, 2]` 的网络为例：

In [ ]:
import numpy as np

def activation_ReLU(x):
    return np.maximum(0, x)

# 5个样本，每个2维特征
inputs = np.array([[0.9, -0.4],
                    [-0.8, 0.5],
                    [-0.5, 0.8],
                    [0.7, -0.3],
                    [-0.9, -0.4]])

# 手动创建每层的权重和偏置
W1 = np.random.randn(2, 3); b1 = np.random.randn(3)
W2 = np.random.randn(3, 4); b2 = np.random.randn(4)
W3 = np.random.randn(4, 2); b3 = np.random.randn(2)

# 逐层前向传播
out1 = activation_ReLU(np.dot(inputs, W1) + b1)
out2 = activation_ReLU(np.dot(out1, W2) + b2)
out3 = activation_ReLU(np.dot(out2, W3) + b3)

print("第1层输出 (5×3):", out1.shape)
print("第2层输出 (5×4):", out2.shape)
print("第3层输出 (5×2):", out3.shape)

## 3.2 Layer 类

把每层的权重、偏置和前向传播封装到一个类中：

**注意：** `__init__` 前后各两个下划线，写成 `_init_` 会导致构造函数不被调用。

In [ ]:
class Layer:
    def __init__(self, n_inputs, n_neurons):
        """初始化层
        
        参数:
            n_inputs: 输入维度
            n_neurons: 该层神经元数量
        """
        self.weights = np.random.randn(n_inputs, n_neurons)
        self.biases = np.random.randn(n_neurons)

    def forward(self, inputs):
        """前向传播：Z = X·W + b"""
        self.sum = np.dot(inputs, self.weights) + self.biases
        return self.sum

# 测试：创建一层，输入2维，输出3个神经元
layer = Layer(2, 3)
test_input = np.array([[1.0, 2.0]])
print("输出:", layer.forward(test_input))
print("权重形状:", layer.weights.shape)  # (2, 3)
print("偏置形状:", layer.biases.shape)   # (3,)

## 3.3 Network 类

`Network` 接收一个形状列表（如 `[2, 3, 4, 2]`），自动创建所有层，并提供统一的前向传播方法。

In [ ]:
class Network:
    def __init__(self, network_shape):
        """根据形状列表创建网络
        
        参数:
            network_shape: 如 [2, 3, 4, 2]
                2个输入 → 3个隐藏神经元 → 4个隐藏神经元 → 2个输出
        """
        self.shape = network_shape
        self.layers = []
        for i in range(len(network_shape) - 1):
            layer = Layer(network_shape[i], network_shape[i + 1])
            self.layers.append(layer)

    def network_forward(self, inputs):
        """依次通过所有层，返回每层的输出"""
        outputs = [inputs]  # outputs[0] = 原始输入
        for i in range(len(self.layers)):
            layer_sum = self.layers[i].forward(outputs[i])
            # 隐藏层用 ReLU，最后一层先不激活（后面会加 Softmax）
            if i < len(self.layers) - 1:
                layer_output = activation_ReLU(layer_sum)
            else:
                layer_output = layer_sum  # 输出层暂不激活
            outputs.append(layer_output)
        return outputs

In [ ]:
# 创建网络：2维输入 → 3 → 4 → 2维输出
net = Network([2, 3, 4, 2])

# 前向传播
inputs = np.array([[0.1, 0.2],
                    [0.3, 0.4],
                    [0.5, 0.6]])
outputs = net.network_forward(inputs)

print("网络结构:", net.shape)
print("层数:", len(net.layers))
for i, out in enumerate(outputs):
    print(f"第{i}层输出 shape={out.shape}:")
    print(out)
    print()

---

## 小结

- `Layer` 类封装了单层的权重、偏置和前向传播
- `Network` 类根据形状列表自动创建多层，并提供统一的前向传播接口
- `__init__` 是 Python 类的构造函数，注意双下划线
- 输出层暂不加激活函数，下一课我们将引入 Softmax

**下一课**我们将实现 Softmax 激活函数，让网络输出概率分布。